<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Ex-Dividend_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import time
from IPython.display import display


## Ex-Dividend Date Price Drop Analysis

**Purpose:** Analyzes stock price behavior (specifically drops in Open and Low prices) relative to the dividend amount on ex-dividend dates for a list of S&P 500 stocks. It also includes market-adjusted drop calculations using SPY as a benchmark.

**Input Data:**
- `sp500_df` (DataFrame): A DataFrame containing S&P 500 stock symbols.
- Historical stock data from Yahoo Finance for individual tickers and SPY benchmark.

**Output Data:**
- `df_ex_dates` (DataFrame): A MultiIndex DataFrame containing detailed ex-dividend event records, including:
    - `Info`: Ticker, Ex_Dividend_Date, Dividend_Amount, Prev_Close, Div_Yield_Pct
    - `Open Metrics`: Ex_Open, Open_Price_Drop, Open_Drop_Pct_of_Div
    - `Low Metrics`: Ex_Low, Low_Price_Drop, Low_Drop_Pct_of_Div
    - `Market Adjusted`: SPY_Adjusted_Drop, Adj_Open_Drop_Pct_of_Div
- `stats_df` (DataFrame): A DataFrame summarizing the mean, median, standard deviation, and count of 'Open Drop (% of Div)' and 'Low Drop (% of Div)'.

**How it works:**
1. **Load S&P 500 Tickers:** Reads a CSV containing S&P 500 company symbols and processes them.
2. **Download Benchmark Data:** Fetches 2 years of historical data for the SPY ETF (benchmark) to calculate market adjustments.
3. **Iterate and Analyze:** For each S&P 500 ticker:
    - Downloads 2 years of historical stock data.
    - Identifies ex-dividend dates where a dividend was paid.
    - For each ex-dividend event:
        - Calculates the dividend yield percentage based on the previous day's closing price.
        - Computes the price drop from the previous close to the ex-dividend open (`Open_Price_Drop`).
        - Computes the price drop from the previous close to the ex-dividend low (`Low_Price_Drop`).
        - Expresses these drops as a percentage of the dividend amount (`Open_Drop_Pct_of_Div`, `Low_Drop_Pct_of_Div`).
        - Calculates a market-adjusted open drop by accounting for the SPY's overnight return on the same day.
        - Stores all calculated metrics in `ex_div_records`.
4. **Aggregate Results:** Concatenates all `ex_div_records` into a final DataFrame `df_ex_dates`.
5. **Summarize Statistics:** Calculates descriptive statistics (mean, median, std dev, count) for the `Open_Drop_Pct_of_Div` and `Low_Drop_Pct_of_Div` columns and displays them in `stats_df`.

In [2]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
SP500_id = '1gWxaB7UZfvHHGlQJoQRucvvO5eGBOCKY'
SP500 = f'https://drive.google.com/uc?export=download&id={SP500_id}'


#1KcMdKzzwwkcc7fLpPAe_cwqkw5TGBv8g
#1gWxaB7UZfvHHGlQJoQRucvvO5eGBOCKY



### Data Collection and Ex-Dividend Event Processing

In [10]:
import pandas as pd
import yfinance as yf

# Read local CSV file
sp500_df = pd.read_csv(SP500)
tickers = sp500_df['Symbol'].astype(str).str.replace('.', '-', regex=False).tolist()

# 1. Download benchmark (SPY) data first for market-adjustment calculation
spy_hist = yf.Ticker("SPY").history(period="2y", auto_adjust=False)
spy_hist.index = spy_hist.index.tz_localize(None) # Normalize timezone

ex_div_records = []

# Process tickers (increase batch size safely)
for ticker in tickers[:503]:
    try:
        stock = yf.Ticker(ticker)
        # FORCE auto_adjust=False to get raw unadjusted transaction prices
        hist = stock.history(period="2y", auto_adjust=False)

        if hist.empty or 'Dividends' not in hist.columns:
            continue

        # Normalize timezones to prevent index lookup failures
        hist.index = hist.index.tz_localize(None)
        div_rows = hist[hist['Dividends'] > 0]

        for date, row in div_rows.iterrows():
            loc = hist.index.get_loc(date)

            # If get_loc returns a slice or mask due to duplicates, extract integer
            if isinstance(loc, slice):
                loc = loc.start
            elif not isinstance(loc, int):
                loc = int(loc[0])

            # Guard against first row edge-case (loc - 1 == -1)
            if loc <= 0:
                continue

            prev_row = hist.iloc[loc - 1]
            prev_close = prev_row['Close']
            ex_open = row['Open']
            ex_low = row['Low']
            ex_close = row['Close']  # Added Ex_Close
            div_amount = row['Dividends']

            if prev_close <= 0 or div_amount <= 0:
                continue

            # Yield calculation relative to previous close
            div_yield_pct = (div_amount / prev_close) * 100

            # --- OPEN DROP METRICS ---
            open_drop = prev_close - ex_open
            open_drop_pct_div = (open_drop / div_amount) * 100

            # --- LOW DROP METRICS ---
            low_drop = prev_close - ex_low
            low_drop_pct_div = (low_drop / div_amount) * 100

            # --- CLOSE DROP METRICS (Added) ---
            close_drop = prev_close - ex_close
            close_drop_pct_div = (close_drop / div_amount) * 100

            # --- MARKET-ADJUSTED OPEN DROP (SPY Benchmark) ---
            if date in spy_hist.index:
                spy_loc = spy_hist.index.get_loc(date)
                if isinstance(spy_loc, slice): spy_loc = spy_loc.start
                if spy_loc > 0:
                    spy_prev_close = spy_hist.iloc[spy_loc - 1]['Close']
                    spy_ex_open = spy_hist.loc[date, 'Open']
                    spy_return = (spy_ex_open - spy_prev_close) / spy_prev_close

                    adjusted_open_drop = open_drop - (prev_close * spy_return)
                    adj_open_drop_pct_div = (adjusted_open_drop / div_amount) * 100
                else:
                    adjusted_open_drop, adj_open_drop_pct_div = None, None
            else:
                adjusted_open_drop, adj_open_drop_pct_div = None, None

            ex_div_records.append({
                ('Info', 'Ticker'): ticker,
                ('Info', 'Ex_Dividend_Date'): date.strftime('%Y-%m-%d'),
                ('Info', 'Dividend_Amount'): div_amount,
                ('Info', 'Prev_Close'): prev_close,
                ('Info', 'Div_Yield_Pct'): div_yield_pct,

                ('Open Metrics', 'Ex_Open'): ex_open,
                ('Open Metrics', 'Open_Price_Drop'): open_drop,
                ('Open Metrics', 'Open_Drop_Pct_of_Div'): open_drop_pct_div,

                ('Low Metrics', 'Ex_Low'): ex_low,
                ('Low Metrics', 'Low_Price_Drop'): low_drop,
                ('Low Metrics', 'Low_Drop_Pct_of_Div'): low_drop_pct_div,

                # Added Close Metrics
                ('Close Metrics', 'Ex_Close'): ex_close,
                ('Close Metrics', 'Close_Price_Drop'): close_drop,
                ('Close Metrics', 'Close_Drop_Pct_of_Div'): close_drop_pct_div,

                ('Market Adjusted', 'SPY_Adjusted_Drop'): adjusted_open_drop,
                ('Market Adjusted', 'Adj_Open_Drop_Pct_of_Div'): adj_open_drop_pct_div,
            })

    except Exception as e:
        print(f"Could not fetch data for {ticker}: {e}")

df_ex_dates = pd.DataFrame(ex_div_records)
df_ex_dates.columns = pd.MultiIndex.from_tuples(df_ex_dates.columns)

print(f"Scanned Tickers: {len(tickers)} / {len(sp500_df)}")
print(f"Total Ex-Dividend Events Captured: {len(df_ex_dates)}")
print(f"Unique Tickers with Dividend Events: {df_ex_dates[('Info', 'Ticker')].nunique() if not df_ex_dates.empty else 0}")

Scanned Tickers: 503 / 503
Total Ex-Dividend Events Captured: 3174
Unique Tickers with Dividend Events: 407


### Summary Statistics of Price Drops

In [11]:
# 1. Select the percentage drop columns
open_pct = df_ex_dates[('Open Metrics', 'Open_Drop_Pct_of_Div')]
low_pct = df_ex_dates[('Low Metrics', 'Low_Drop_Pct_of_Div')]

# 2. Compute summary statistics
stats_data = {
    'Open Drop (% of Div)': {
        'Mean': open_pct.mean(),
        'Median': open_pct.median(),
        'Std Dev': open_pct.std(),
        'Count': open_pct.count()
    },
    'Low Drop (% of Div)': {
        'Mean': low_pct.mean(),
        'Median': low_pct.median(),
        'Std Dev': low_pct.std(),
        'Count': low_pct.count()
    }
}

# 3. Convert to DataFrame and format
stats_df = pd.DataFrame(stats_data)

# Round values for clean display
display(stats_df.round(2))

,Open Drop (% of Div),Low Drop (% of Div)
Mean,52.46,603.65
Median,89.68,271.50
Std Dev,1519.91,1899.27
Count,3174.00,3174.00


In [12]:
# 1. Trim extreme 1% outliers to isolate structural behavior
open_pct = df_ex_dates[('Open Metrics', 'Open_Drop_Pct_of_Div')]
q_low = open_pct.quantile(0.01)
q_high = open_pct.quantile(0.99)

df_trimmed = df_ex_dates[
    (open_pct >= q_low) & (open_pct <= q_high)
].copy()

# 2. Segment by Dividend Yield Buckets
df_trimmed['Yield_Bucket'] = pd.cut(
    df_trimmed[('Info', 'Div_Yield_Pct')],
    bins=[0, 0.25, 0.50, 1.0, 100.0],
    labels=['< 0.25%', '0.25% - 0.50%', '0.50% - 1.00%', '> 1.00%']
)

# 3. Aggregate Drop Performance by Yield Tier
# Pass target_col inside a list [ [tuple] ] to keep Pandas happy with MultiIndex
target_col = [('Open Metrics', 'Open_Drop_Pct_of_Div')]

yield_summary = (
    df_trimmed
    .groupby('Yield_Bucket', observed=False)[target_col]
    .agg(['mean', 'median', 'std', 'count'])
)

# Flatten MultiIndex columns for clean display
yield_summary.columns = ['Mean_Drop_Pct', 'Median_Drop_Pct', 'Std_Dev', 'Event_Count']

print("--- OPEN DROP BY DIVIDEND YIELD TIER (Middle 98% of Events) ---")
display(yield_summary.round(2))

--- OPEN DROP BY DIVIDEND YIELD TIER (Middle 98% of Events) ---


,Mean_Drop_Pct,Median_Drop_Pct,Std_Dev,Event_Count
Yield_Bucket,,,,
< 0.25%,66.29,62.50,723.11,649
0.25% - 0.50%,80.11,79.55,268.67,913
0.50% - 1.00%,86.24,91.67,143.20,1148
> 1.00%,94.17,99.19,69.24,400


In [15]:
# 1. Assign Dividend Yield Buckets
df_ex_dates['Yield_Bucket'] = pd.cut(
    df_ex_dates[('Info', 'Div_Yield_Pct')],
    bins=[0, 0.25, 0.50, 1.0, 100.0],
    labels=['< 0.25%', '0.25% - 0.50%', '0.50% - 1.00%', '> 1.00%']
)

# Define columns
low_col = ('Low Metrics', 'Low_Drop_Pct_of_Div')
close_col = ('Close Metrics', 'Close_Drop_Pct_of_Div')

# Function to compute stats per yield bucket
def analyze_low_and_close(group):
    # 25th percentile of Low Drop % = minimum drop reached by the top 75% deepest dropping events
    p25_threshold = group[low_col].quantile(0.25)

    # Filter subset that failed to drop at least to this threshold
    failed_subset = group[group[low_col] < p25_threshold]

    return pd.Series({
        'Event Count': len(group),
        'Min Drop for Top 75% (Low Drop % of Div)': p25_threshold,
        'Failed Subgroup Count (< Top 75% Drop)': len(failed_subset),
        'Failed Subgroup Mean Close Drop (% of Div)': failed_subset[close_col].mean(),
        'Failed Subgroup Median Close Drop (% of Div)': failed_subset[close_col].median(),
        'All Events Median Close Drop (% of Div)': group[close_col].median()
    })

# 2. Run analysis grouped by Dividend Yield Bucket
low_drop_analysis = (
    df_ex_dates
    .groupby('Yield_Bucket', observed=False)
    .apply(analyze_low_and_close)
)

print("--- LOW DROP THRESHOLD & CLOSING PERFORMANCE BY YIELD TIER ---")
display(low_drop_analysis.round(2))

--- LOW DROP THRESHOLD & CLOSING PERFORMANCE BY YIELD TIER ---


/tmp/ipykernel_3973/3513017986.py:35: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  .apply(analyze_low_and_close)
/tmp/ipykernel_3973/3513017986.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(analyze_low_and_close)


,Event Count,Min Drop for Top 75% (Low Drop % of Div),Failed Subgroup Count (< Top 75% Drop),Failed Subgroup Mean Close Drop (% of Div),Failed Subgroup Median Close Drop (% of Div),All Events Median Close Drop (% of Div)
Yield_Bucket,,,,,,
< 0.25%,712.0,334.28,178.0,-2375.93,-697.77,196.58
0.25% - 0.50%,913.0,174.00,228.0,-346.17,-242.87,97.00
0.50% - 1.00%,1149.0,133.72,287.0,-132.68,-80.95,85.71
> 1.00%,400.0,119.65,100.0,-46.66,-20.17,88.68


In [16]:
# 1. Define Long Strategy Metrics
# Dividend Received = +100% of Dividend
# Capital Loss/Gain = -(Close_Drop_Pct_of_Div)
# Net P&L (% of Div) = 100 - Close_Drop_Pct_of_Div
df_ex_dates[('Long Strategy', 'Net_PnL_Pct_of_Div')] = 100.0 - df_ex_dates[('Close Metrics', 'Close_Drop_Pct_of_Div')]

# Net Return on Invested Capital (%) = (Net P&L / Prev_Close) * 100
# Equivalent to: Div_Yield_Pct * (Net_PnL_Pct_of_Div / 100)
df_ex_dates[('Long Strategy', 'Return_on_Capital_Pct')] = (
    df_ex_dates[('Info', 'Div_Yield_Pct')] * (df_ex_dates[('Long Strategy', 'Net_PnL_Pct_of_Div')] / 100.0)
)

# Trade Outcome: Win if Net P&L > 0
df_ex_dates[('Long Strategy', 'Is_Profitable')] = df_ex_dates[('Long Strategy', 'Net_PnL_Pct_of_Div')] > 0

# 2. Aggregation Function
def analyze_long_strategy(group):
    total_trades = len(group)
    wins = group[('Long Strategy', 'Is_Profitable')].sum()
    win_rate = (wins / total_trades) * 100 if total_trades > 0 else 0

    net_pnl_div = group[('Long Strategy', 'Net_PnL_Pct_of_Div')]
    roc = group[('Long Strategy', 'Return_on_Capital_Pct')]

    return pd.Series({
        'Total Trades': total_trades,
        'Win Rate (%)': win_rate,
        'Median Net PnL (% of Div)': net_pnl_div.median(),
        'Mean Net PnL (% of Div)': net_pnl_div.mean(),
        'Median Return on Capital (%)': roc.median(),
        'Mean Return on Capital (%)': roc.mean(),
        'Max Win (% of Div)': net_pnl_div.max(),
        'Max Loss (% of Div)': net_pnl_div.min()
    })

# 3. Execute Analysis Grouped by Yield Bucket
long_analysis = (
    df_ex_dates
    .groupby('Yield_Bucket', observed=False)
    .apply(analyze_long_strategy)
)

print("--- OVERNIGHT LONG STRATEGY ANALYSIS (Buy Prev Close -> Sell Ex Close + Collect Div) ---")
display(long_analysis.round(2))

--- OVERNIGHT LONG STRATEGY ANALYSIS (Buy Prev Close -> Sell Ex Close + Collect Div) ---


/tmp/ipykernel_3973/1654049211.py:40: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  .apply(analyze_long_strategy)
/tmp/ipykernel_3973/1654049211.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(analyze_long_strategy)


,Total Trades,Win Rate (%),Median Net PnL (% of Div),Mean Net PnL (% of Div),Median Return on Capital (%),Mean Return on Capital (%),Max Win (% of Div),Max Loss (% of Div)
Yield_Bucket,,,,,,,,
< 0.25%,712.0,46.07,-96.58,-5.19,-0.13,-0.10,69899.96,-42420.00
0.25% - 0.50%,913.0,50.05,3.00,-11.13,0.01,-0.05,2904.94,-2218.33
0.50% - 1.00%,1149.0,53.70,14.29,16.91,0.11,0.11,1432.14,-2960.71
> 1.00%,400.0,55.25,11.32,16.60,0.15,0.21,637.21,-928.90
